<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

For item $i$,

$$
Y_i=
\begin{cases}
1, & \text{if item }i\text{ is answered correctly},\\
0, & \text{otherwise}.
\end{cases}
$$

The two-parameter logistic model is

$$
p_i(\theta)
=
P(Y_i=1\mid \Theta=\theta)
=
\frac{1}{1+\exp[-a_i(\theta-b_i)]},
$$

where $a_i>0$ is the discrimination parameter and $b_i$ is the difficulty parameter.

The initial prior is

$$
\Theta\sim\mathcal{N}(0,1),
$$

with density

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$



## 1. Visualizing the Mechanics

The difficulty parameter $b_i$ controls the horizontal position of the item-response curve. Since

$$
p_i(b_i)=\frac{1}{2},
$$

increasing $b_i$ shifts the curve to the right, while decreasing $b_i$ shifts it to the left.

The discrimination parameter $a_i$ controls the steepness. A larger $a_i$ produces a steeper curve near $\theta=b_i$, while a smaller $a_i$ produces a flatter curve.


In [1]:

import numpy as np
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta = np.linspace(-6, 6, 500)

curves = [
    {"a": 0.5, "b": 0.0},
    {"a": 1.5, "b": -2.0},
    {"a": 1.5, "b": 0.0},
    {"a": 1.5, "b": 2.0},
]

fig = go.Figure()

for item in curves:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=p_i(theta, item["a"], item["b"]),
            mode="lines",
            name=f"a={item['a']}, b={item['b']}"
        )
    )

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Latent ability θ",
    yaxis_title="P(Yᵢ=1 | Θ=θ)",
    template="plotly_white"
)

fig.show()



### Interpretation

For fixed $a_i=1.5$, the curves with $b_i=-2,0,2$ have the same shape but different horizontal positions. The curve moves right as $b_i$ increases.

For fixed $b_i=0$, the curve with $a_i=1.5$ is steeper than the curve with $a_i=0.5$. Therefore, the high-discrimination item provides more information about ability values close to its difficulty.



## 2. Sequential Likelihood Contribution

At step $k$, the response is $y_k\in\{0,1\}$. Its likelihood contribution is

$$
\boxed{
L(y_k\mid\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
}
$$

If $y_k=1$,

$$
L(y_k\mid\theta)=p_k(\theta).
$$

If $y_k=0$,

$$
L(y_k\mid\theta)=1-p_k(\theta).
$$

Assuming conditional independence, the joint likelihood for

$$
y^{(k)}=(y_1,\ldots,y_k)
$$

is

$$
\boxed{
L(y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}
}
$$



## 3. Mathematical Formulation of the Running Update

The posterior at step $k-1$ becomes the prior for step $k$. Therefore,

$$
\boxed{
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)})
}
$$

The normalized posterior is

$$
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
=
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)})
}{
\int_{\mathbb{R}}
[p_k(s)]^{y_k}
[1-p_k(s)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}(s\mid y^{(k-1)})
\,ds
}.
$$



## 4. Dynamic Shifting

For a correct response, $y_k=1$, the update becomes

$$
f_k(\theta)\propto p_k(\theta)f_{k-1}(\theta).
$$

If $b_k$ is large, then $p_k(\theta)$ is very small for low values of $\theta$ and becomes large only for larger ability values. Multiplying the previous posterior by this likelihood reduces posterior mass at low ability values and increases the relative mass at high ability values.

Therefore, a correct answer to a difficult item shifts the posterior peak toward larger values of $\theta$.



## 5. Tracking Certainty and Sharpness

A large $a_k$ produces a steep likelihood curve. This means the response strongly distinguishes between nearby ability values, especially around $\theta=b_k$. Therefore, the posterior usually becomes sharper and its variance decreases more strongly.

A small $a_k$ produces a flat likelihood curve. The response provides less information, so the posterior changes only slightly and remains broader.

Thus:

$$
\text{large }a_k
\Rightarrow
\text{more information and sharper posterior},
$$

while

$$
\text{small }a_k
\Rightarrow
\text{less information and broader posterior}.
$$



## 6. Numerical Implementation of a Running Grid

Choose a fixed grid

$$
\theta_1,\theta_2,\ldots,\theta_M.
$$

At step $k$:

1. Evaluate

$$
p_k(\theta_m)
=
\frac{1}{1+\exp[-a_k(\theta_m-b_k)]}.
$$

2. Compute

$$
L_k(\theta_m)
=
[p_k(\theta_m)]^{y_k}
[1-p_k(\theta_m)]^{1-y_k}.
$$

3. Form the unnormalized posterior

$$
\widetilde{P}_k(\theta_m)
=
P_{k-1}(\theta_m)L_k(\theta_m).
$$

4. Compute the normalizing constant using the trapezoidal rule:

$$
Z_k
\approx
\operatorname{trapezoid}
(\widetilde{P}_k,\theta).
$$

5. Normalize:

$$
P_k(\theta_m)
=
\frac{\widetilde{P}_k(\theta_m)}{Z_k}.
$$

6. Compute the posterior mean:

$$
\widehat{\theta}^{(k)}_{\text{Bayes}}
\approx
\operatorname{trapezoid}
(\theta P_k,\theta).
$$

7. Compute the MAP estimate:

$$
\widehat{\theta}^{(k)}_{\text{MAP}}
=
\theta_{\arg\max_m P_k(\theta_m)}.
$$


In [2]:

import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_grid = np.linspace(-5, 5, 1000)

posterior = norm.pdf(theta_grid, 0, 1)
posterior /= np.trapezoid(posterior, theta_grid)

items = [
    {"a": 1.0, "b": -1.5, "y": 1},
    {"a": 1.5, "b": 0.5, "y": 1},
    {"a": 1.2, "b": 1.5, "y": 0},
    {"a": 2.0, "b": 0.2, "y": 1},
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theta_grid,
    y=posterior,
    mode="lines",
    name="Initial prior"
))

for step, item in enumerate(items, start=1):
    prob = p_i(theta_grid, item["a"], item["b"])
    likelihood = prob**item["y"] * (1-prob)**(1-item["y"])

    posterior *= likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    fig.add_trace(go.Scatter(
        x=theta_grid,
        y=posterior,
        mode="lines",
        name=f"Step {step}"
    ))

fig.update_layout(
    title="Sequential Bayesian Ability Updating",
    xaxis_title="θ",
    yaxis_title="Posterior density",
    template="plotly_white"
)

fig.show()



## 7. Evaluating Convergence over the Timeline

Let

$$
\theta_{\text{true}}=0.75,
\qquad
n=20.
$$

Generate

$$
a_k\sim\operatorname{Uniform}(0.5,2.0)
$$

and

$$
b_k\sim\mathcal{N}(0,1).
$$

At each step, simulate

$$
y_k=
\begin{cases}
1, & U_k<p_k(\theta_{\text{true}}),\\
0, & \text{otherwise},
\end{cases}
$$

where $U_k\sim\operatorname{Uniform}(0,1)$.


In [3]:

import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

rng = np.random.default_rng(42)

theta_true = 0.75
n = 20
theta_grid = np.linspace(-5, 5, 1500)

a_values = rng.uniform(0.5, 2.0, n)
b_values = rng.normal(0, 1, n)

posterior = norm.pdf(theta_grid, 0, 1)
posterior /= np.trapezoid(posterior, theta_grid)

bayes_estimates = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]
steps = list(range(n + 1))

for k in range(n):
    p_true = p_i(theta_true, a_values[k], b_values[k])
    y_k = int(rng.uniform(0, 1) < p_true)

    p_grid = p_i(theta_grid, a_values[k], b_values[k])
    likelihood = p_grid**y_k * (1-p_grid)**(1-y_k)

    posterior *= likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    bayes_estimates.append(
        np.trapezoid(theta_grid * posterior, theta_grid)
    )
    map_estimates.append(
        theta_grid[np.argmax(posterior)]
    )

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps,
    y=bayes_estimates,
    mode="lines+markers",
    name="Posterior Mean"
))

fig.add_trace(go.Scatter(
    x=steps,
    y=map_estimates,
    mode="lines+markers",
    name="MAP Estimate"
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability = 0.75"
)

fig.update_layout(
    title="Convergence of Ability Estimates",
    xaxis_title="Item step k",
    yaxis_title="Estimated ability",
    template="plotly_white"
)

fig.show()



### Analysis

At the beginning, the estimates can fluctuate because only a small number of responses have been observed. As $k$ increases, more evidence is accumulated, so the posterior generally becomes narrower and the estimators tend to move closer to $\theta_{\text{true}}$.

The posterior mean and MAP estimate may differ when the posterior is asymmetric. A decreasing distance from $\theta_{\text{true}}$ indicates improving accuracy, while a narrower posterior indicates increasing confidence.



# Q. Bayesian Tracking of Click-Through Rates via Conjugate Beta-Binomial Updates

Let $\Theta=\theta$ be the unknown click-through rate, where

$$
\theta\in[0,1].
$$

At step $k$,

$$
Y_k=
\begin{cases}
1, & \text{if the user clicks},\\
0, & \text{if the user does not click}.
\end{cases}
$$

The model is

$$
Y_k\mid\Theta=\theta
\sim
\operatorname{Bernoulli}(\theta).
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(\alpha_0,\beta_0),
$$

with density

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(\alpha_0,\beta_0)}
\theta^{\alpha_0-1}
(1-\theta)^{\beta_0-1}.
$$



## 1. Structural Probability and Properties

For

$$
\Theta\sim\operatorname{Beta}(\alpha,\beta),
$$

the density is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}(1-\theta)^{\beta-1}.
$$

The mean is

$$
\mathbb{E}[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Therefore:

$$
\operatorname{Beta}(1,1)
\Rightarrow
\mathbb{E}[\Theta]=0.5,
$$

$$
\operatorname{Beta}(2,8)
\Rightarrow
\mathbb{E}[\Theta]=0.2,
$$

and

$$
\operatorname{Beta}(8,2)
\Rightarrow
\mathbb{E}[\Theta]=0.8.
$$

Increasing $\alpha$ relative to $\beta$ shifts the density toward $1$, while increasing $\beta$ relative to $\alpha$ shifts it toward $0$.


In [ ]:

import numpy as np
from scipy.stats import beta
import plotly.graph_objects as go

theta = np.linspace(0.001, 0.999, 800)

configs = [
    (1, 1),
    (2, 8),
    (8, 2),
]

fig = go.Figure()

for a, b in configs:
    fig.add_trace(go.Scatter(
        x=theta,
        y=beta.pdf(theta, a, b),
        mode="lines",
        name=f"Beta({a},{b})"
    ))

fig.update_layout(
    title="Beta Density Functions",
    xaxis_title="CTR θ",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()



## 2. Sequential Likelihood and Joint History

For a single response,

$$
\boxed{
L(y_k\mid\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}
}
$$

For the running history

$$
y^{(k)}=(y_1,\ldots,y_k),
$$

the joint likelihood is

$$
L(y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}(1-\theta)^{1-y_i}.
$$

Let

$$
C_k=\sum_{i=1}^{k}y_i
$$

be the number of clicks. Then

$$
\boxed{
L(y^{(k)}\mid\theta)
=
\theta^{C_k}(1-\theta)^{k-C_k}
}
$$



## 3. Closed-Form Analytical Updates

Suppose

$$
\Theta\mid Y^{(k-1)}
\sim
\operatorname{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

Then

$$
f_{k-1}(\theta)
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

After observing $y_k$,

$$
f_k(\theta)
\propto
\theta^{y_k}(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Therefore,

$$
f_k(\theta)
\propto
\theta^{(\alpha_{k-1}+y_k)-1}
(1-\theta)^{(\beta_{k-1}+1-y_k)-1}.
$$

Hence,

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+(1-y_k)
}
$$

so

$$
\boxed{
\Theta\mid Y^{(k)}
\sim
\operatorname{Beta}(\alpha_k,\beta_k)
}
$$

After $k$ observations,

$$
\alpha_k=\alpha_0+C_k,
$$

and

$$
\beta_k=\beta_0+k-C_k.
$$

The posterior mean is

$$
\boxed{
\mathbb{E}[\Theta\mid Y^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}
}
$$



## 4. Dynamic Shifting Mechanics

If $y_k=1$,

$$
\alpha_k=\alpha_{k-1}+1,
\qquad
\beta_k=\beta_{k-1},
$$

so the posterior shifts toward larger values of $\theta$.

If $y_k=0$,

$$
\alpha_k=\alpha_{k-1},
\qquad
\beta_k=\beta_{k-1}+1,
$$

so the posterior shifts toward smaller values of $\theta$.

This model is conjugate because the posterior remains in the Beta family. In contrast, the 2PL model is non-conjugate, so a numerical grid or another approximation method is required.



## 5. Running Point Estimators

The running posterior mean is

$$
\boxed{
\widehat{\theta}^{(k)}_{\text{Bayes}}
=
\frac{\alpha_k}{\alpha_k+\beta_k}
}
$$

When $\alpha_k>1$ and $\beta_k>1$, the MAP estimate is

$$
\boxed{
\widehat{\theta}^{(k)}_{\text{MAP}}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
}
$$



## 6. Performance Tracking and Convergence Analysis

Let

$$
\theta_{\text{true}}=0.35,
\qquad
n=100,
\qquad
\alpha_0=\beta_0=1.
$$

At each step,

$$
y_k=
\begin{cases}
1, & U_k<0.35,\\
0, & \text{otherwise},
\end{cases}
$$

where

$$
U_k\sim\operatorname{Uniform}(0,1).
$$


In [4]:

import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(42)

theta_true = 0.35
n = 100

alpha_k = 1
beta_k = 1

steps = list(range(n + 1))
posterior_means = [alpha_k / (alpha_k + beta_k)]
map_estimates = [0.5]

for k in range(1, n + 1):
    y_k = int(rng.uniform(0, 1) < theta_true)

    alpha_k += y_k
    beta_k += 1 - y_k

    posterior_mean = alpha_k / (alpha_k + beta_k)

    if alpha_k > 1 and beta_k > 1:
        map_estimate = (
            (alpha_k - 1) /
            (alpha_k + beta_k - 2)
        )
    elif alpha_k <= 1 and beta_k > 1:
        map_estimate = 0.0
    elif alpha_k > 1 and beta_k <= 1:
        map_estimate = 1.0
    else:
        map_estimate = 0.5

    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps,
    y=posterior_means,
    mode="lines",
    name="Posterior Mean"
))

fig.add_trace(go.Scatter(
    x=steps,
    y=map_estimates,
    mode="lines",
    name="MAP Estimate"
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35"
)

fig.update_layout(
    title="Sequential CTR Estimation",
    xaxis_title="Number of impressions",
    yaxis_title="Estimated CTR",
    template="plotly_white"
)

fig.show()

print(f"Final posterior: Beta({alpha_k}, {beta_k})")
print(f"Final posterior mean: {posterior_means[-1]:.4f}")
print(f"Final MAP estimate: {map_estimates[-1]:.4f}")


Final posterior: Beta(34, 68)
Final posterior mean: 0.3333
Final MAP estimate: 0.3300



### Analysis

At small sample sizes, the prior has a noticeable effect. After $k$ observations,

$$
\widehat{\theta}^{(k)}_{\text{Bayes}}
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}.
$$

As $k$ increases, the fixed prior parameters become small compared with the accumulated click and non-click counts. Therefore,

$$
\widehat{\theta}^{(k)}_{\text{Bayes}}
\approx
\frac{C_k}{k}.
$$

By the law of large numbers,

$$
\frac{C_k}{k}
\to
\theta_{\text{true}}.
$$

Thus, the estimators generally move closer to $0.35$ as more impressions are observed. The posterior variance decreases, showing that the platform becomes increasingly confident about the advertisement's true CTR.
